![GovSpace Logo](https://govspace.io/wp-content/uploads/2025/09/GovSpace-web.svg)

# 🤖 Basic Agent with Tool Use

> **💡 Tip:** Copy this notebook into your personal folder (e.g. `matt-grasser/`) before editing. The `TEMPLATES` folder syncs from GitHub on each login and is read-only.

In this notebook you'll build a simple agent that can **call tools** (functions) to accomplish tasks.

This is the foundation of agentic AI: an LLM that can take actions, not just generate text.

## What you'll learn
1. How to define tools that Claude can call
2. The tool-use conversation loop
3. How to handle tool results and multi-turn interactions

In [ ]:
import os
import anthropic

# Keys should already be set from 00-Setup. If not, set them here:
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

client = anthropic.Anthropic()

## Step 1: Define Tools

Tools are JSON schemas that tell the model what functions are available and what arguments they accept.

Let's create two simple tools: a calculator and a weather lookup.

In [ ]:
tools = [
    {
        "name": "calculate",
        "description": "Evaluate a mathematical expression. Use this for any arithmetic.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A mathematical expression to evaluate, e.g. '(25 * 4) + 10'"
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "City name, e.g. 'Ottawa'"
                }
            },
            "required": ["city"]
        }
    }
]

## Step 2: Implement Tool Handlers

These are the actual Python functions that run when the model calls a tool.

In [ ]:
def handle_tool_call(tool_name: str, tool_input: dict) -> str:
    """Dispatch a tool call to the appropriate handler."""
    if tool_name == "calculate":
        try:
            # Safety note: in production, use a proper math parser, not eval()
            result = eval(tool_input["expression"])
            return str(result)
        except Exception as e:
            return f"Error: {e}"

    elif tool_name == "get_weather":
        # Simulated weather data (in a real app, call a weather API)
        fake_weather = {
            "ottawa": "☀️ 22°C, sunny",
            "toronto": "🌤️ 19°C, partly cloudy",
            "vancouver": "🌧️ 14°C, rain",
        }
        city = tool_input["city"].lower()
        return fake_weather.get(city, f"🌡️ 20°C, weather data not available for {tool_input['city']}")

    return f"Unknown tool: {tool_name}"

## Step 3: The Agent Loop

The core pattern: send a message → if the model wants to call a tool → execute it → feed the result back → repeat until the model responds with text.

This is the **agentic loop** — the model decides what to do next.

In [ ]:
def run_agent(user_message: str, max_turns: int = 5) -> str:
    """Run a simple tool-use agent loop."""
    messages = [{"role": "user", "content": user_message}]

    for turn in range(max_turns):
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )

        # If the model just responds with text, we're done
        if response.stop_reason == "end_turn":
            text_blocks = [b.text for b in response.content if b.type == "text"]
            return "\n".join(text_blocks)

        # If the model wants to use tools, execute them
        if response.stop_reason == "tool_use":
            # Add the assistant's response (with tool_use blocks) to messages
            messages.append({"role": "assistant", "content": response.content})

            # Process each tool call
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 Calling: {block.name}({block.input})")
                    result = handle_tool_call(block.name, block.input)
                    print(f"  📎 Result: {result}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })

            # Feed results back to the model
            messages.append({"role": "user", "content": tool_results})

    return "Agent reached max turns without completing."

## Step 4: Try It Out

Ask the agent questions that require tool use:

In [ ]:
# A question that requires the calculator
print("--- Calculator ---")
result = run_agent("What is 47 * 89 + 123?")
print(f"\n{result}")

In [ ]:
# A question that requires weather lookup
print("--- Weather ---")
result = run_agent("What's the weather like in Ottawa and Vancouver?")
print(f"\n{result}")

In [ ]:
# A question that requires BOTH tools
print("--- Multi-tool ---")
result = run_agent(
    "If it's 22°C in Ottawa, what is that in Fahrenheit? "
    "Also check: what's the actual weather in Toronto?"
)
print(f"\n{result}")

## 💡 Key Takeaways

1. **Tools are schemas** — You describe what's available; the model decides when to use them
2. **The agent loop** — Send → tool call → execute → feed back → repeat
3. **The model orchestrates** — It chooses which tools to call and in what order
4. **Multi-tool calls** — The model can call multiple tools in a single turn

---

<div style="background-color:#507dcb; color:white; padding:10px; border-radius:5px; margin-bottom:5px;"><strong>→ Next:</strong> 02-LangGraph-Workflow.ipynb — Build a stateful, multi-step workflow where agents can branch, loop, and hand off to each other</div>
<div style="background-color:#507dcb; color:white; padding:10px; border-radius:5px; margin-bottom:5px;"><strong>→ Then:</strong> 03-Graduated-Autonomy.ipynb — Human-in-the-loop graduated autonomy</div>